# Salary Prediction — Full Training Run

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv

### If you hit a numpy/scipy import error (`_ARRAY_API not found`, `AttributeError` from `scipy`, a `UserWarning` about needing `numpy<1.23`, etc.)


In [ ]:

%pip install --upgrade --force-reinstall numpy scipy

## 2. One-time setup — `.env` and the raw CSV

Idempotent: skips anything already done on a previous run.

In [ ]:
import subprocess

if not os.path.exists(".env"):
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")
else:
    print(".env already exists (not overwritten).")

print()
print(open(".env").read())

In [ ]:
if not os.path.exists("data/raw/survey_results_public.csv"):
    result = subprocess.run(
        ["unzip", "-o", "data.zip", "survey_results_public.csv", "-d", "data/raw/"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print("data/raw/survey_results_public.csv already exists.")

## 3. Spark session — force a fresh one so every setting actually applies

In [ ]:
from pyspark.sql import SparkSession

try:
    SparkSession.builder.getOrCreate().stop()
    print("Stopped an existing Spark session.")
except Exception as exc:
    print("No existing session to stop (or stop failed) - continuing:", exc)

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryModelTraining")

actual = {
    "spark.driver.memory": spark.sparkContext.getConf().get("spark.driver.memory"),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "spark.default.parallelism (actual cluster parallelism)": spark.sparkContext.defaultParallelism,
    "spark.master": spark.sparkContext.master,
}

print("Expected (from config/settings.py, i.e. your .env):")
print("  SPARK_DRIVER_MEMORY      =", settings.SPARK_DRIVER_MEMORY)
print("  SPARK_SHUFFLE_PARTITIONS =", settings.SPARK_SHUFFLE_PARTITIONS)
print("  TUNING_PARALLELISM       =", settings.TUNING_PARALLELISM)
print()
print("Actual Spark session configuration:")
for key, value in actual.items():
    print(f"  {key} = {value}")
print()

if actual["spark.driver.memory"] != settings.SPARK_DRIVER_MEMORY:
    print("*** MISMATCH: driver memory did NOT apply. ***")
    print("This means a Spark session was already running before this cell ran, and ")
    print("stopping it here wasn't enough to free up a truly fresh JVM. Go to ")
    print("Kernel > Restart, then re-run this notebook from the top cell before ")
    print("continuing to the training cell below.")
else:
    print("Spark session configured correctly - safe to proceed to training.")

## 4. Run the full training pipeline

In [ ]:
from src.training.evaluate_model import run_training_pipeline

run_training_pipeline()

## 5. Results

In [ ]:
import json

print("--- models/model_comparison.csv ---")
print(open("models/model_comparison.csv").read())

print("--- models/model_metadata.json ---")
print(json.dumps(json.load(open("models/model_metadata.json")), indent=2))

print("--- models/model_metrics.json ---")
print(json.dumps(json.load(open("models/model_metrics.json")), indent=2))